<a href="https://colab.research.google.com/github/EgzonnOsmanaj/MesoAI/blob/main/EU_AI_ACT_full_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG System for EU AI Act
---

## 1. Domain, Data Sourcing & Justification

### Domain
This project builds a Retrieval-Augmented Generation (RAG) system over the **EU AI Act** (Regulation (EU) 2024/1689), the landmark European legislation that entered into force on 1 August 2024. The full text runs to ~144 pages of dense legal language organised into recitals, articles, annexes and definitions.

### Data Source
The official PDF is downloaded directly from EUR-Lex, the EU's public legal database:  
`https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689`  
This is a publicly available, authoritative source with no licensing restrictions on use.

### Why RAG Rather Than Plain Prompting?

1. **Recency.** The EU AI Act was published in July 2024. Most LLM training corpora have a knowledge cutoff before the final text was adopted; the model may have seen drafts but not the enacted regulation with correct article numbers, annex references and application dates.
2. **Precision.** Legal answers require exact citation of articles, recitals and annexes. A model generating from parametric memory is prone to hallucinating article numbers or conflating provisions from earlier legislative drafts.
3. **Length & specificity.** The Act defines ~60 technical terms, lists 8 high-risk application areas, 7 prohibited practices, 2 GPAI tiers, and a tiered application timeline. No practical prompt can surface all relevant passages for an arbitrary legal question without retrieval.
4. **Verifiability.** Regulators and compliance officers need page-level citations. RAG grounds every answer in retrieved passages, making outputs auditable in a way that pure generation cannot be.

RAG is therefore the correct architectural choice: it combines the language understanding of an LLM with faithful grounding in the authoritative text.


## 2. Setup & Imports

In [ ]:
!pip -q install pandas numpy scikit-learn sentence-transformers faiss-cpu rank-bm25 pymupdf requests beautifulsoup4 lxml google-genai tqdm

In [ ]:
import os
import re
import json
import time
import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dataclasses import dataclass
import fitz
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import google.generativeai as genai
from google.colab import userdata
from transformers import AutoTokenizer

In [ ]:
DATA_DIR = "data"
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
EVAL_DIR = os.path.join(DATA_DIR, "eval")
for d in [DATA_DIR, RAW_DIR, PROCESSED_DIR, EVAL_DIR]:
    os.makedirs(d, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "eu_ai_act.pdf")
PDF_URL = "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689"

if not os.path.exists(PDF_PATH):
    print(f"Downloading PDF from {PDF_URL} to {PDF_PATH}")
    response = requests.get(PDF_URL)
    response.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        f.write(response.content)
    print("Download complete.")

# Model assignment by task
GEMINI_MODEL = "gemini-3.1-flash-lite"
GEMINI_EVAL_MODEL = "gemini-3.1-flash-lite"

EMBED_MODEL_NAME = "intfloat/e5-base-v2"
RERANK_MODEL_NAME = "BAAI/bge-reranker-base"

TOP_K_BM25   = 20
TOP_K_DENSE  = 20
TOP_K_FUSION = 30
TOP_K_RERANK = 10

In [ ]:
# Fetch the Gemini API key from Colab secrets
GEMINI_API_KEY = userdata.get('GoogleAIAPI')

if not GEMINI_API_KEY:
    raise ValueError(
        "Please set GEMINI_API_KEY in your environment or "
        "Colab secrets (named 'GoogleAIAPI') first."
    )


genai.configure(api_key=GEMINI_API_KEY)
print("Gemini API configured successfully.")


## 3. Preprocessing: Chunking & Embedding

### Chunking Strategy: Baseline vs. Article-Boundary

This notebook implements **two chunking strategies** and evaluates both:

| Strategy | How it works | Why |
|---|---|---|
| **Fixed-size (baseline chunker)** | Splits every 1 800 characters with 200-char overlap, regardless of document structure | Simple reference point; common default |
| **Article-boundary (semantic chunker)** | Splits on `Article X`, `Recital (X)`, and Annex headings detected by regex | Each chunk = one coherent legal provision; no article is split across chunks |

The EU AI Act has clear structural markers (`Article 1`, `Article 2 ... Recital (1)` etc.) making it ideal for boundary-aware chunking. A chunk that starts mid-article loses the legal context that makes the provision meaningful. Article-boundary chunks are longer on average but self-contained, which directly benefits retrieval precision.

Both chunkers produce `Chunk` objects with identical metadata (`doc_id`, `page`, `part`) so the rest of the pipeline is unchanged. The enhanced pipeline uses article-boundary chunks; the baseline pipeline uses fixed-size chunks, preserving a fair comparison.


In [ ]:
from typing import Dict, Any
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    text: str
    metadata: Dict[str, Any]

def clean_text(text):
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [ ]:
TITLE_RE = re.compile(
    r"\bTITLE\s+([IVXLC]+)\b",
    re.IGNORECASE
)

CHAPTER_RE = re.compile(
    r"\bChapter\s+([IVXLC\d]+)\b",
    re.IGNORECASE
)

SECTION_RE = re.compile(
    r"\bSection\s+(\d+)\b",
    re.IGNORECASE
)

ARTICLE_RE = re.compile(
    r"\bArticle\s+(\d+)\b",
    re.IGNORECASE
)

RECITAL_RE = re.compile(
    r"\bRecital\s+\((\d+)\)",
    re.IGNORECASE
)

ANNEX_RE = re.compile(
    r"\bANNEX\s+([IVXLC]+)\b",
    re.IGNORECASE
)

In [ ]:
BOUNDARY_RE = re.compile(
    r'(?=\bArticle\s+\d+\b'
    r'|\bRecital\s+\(\d+\)\b'
    r'|\bANNEX\s+[IVXLC]+\b)',
    re.IGNORECASE
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    EMBED_MODEL_NAME
)

In [ ]:
def split_long_text_tokens(
    text,
    tokenizer,
    max_tokens=400,
    overlap_tokens=50
):

    # IMPORTANT: disable special tokens for consistent counting
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0

    while start < len(tokens):

        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]

        chunk_text = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        ).strip()

        # HARD SAFETY CHECK (prevents 512+ surprises later)
        final_len = len(tokenizer.encode(chunk_text, add_special_tokens=True))

        if final_len > 512:
            # fallback: aggressively trim until safe
            chunk_tokens = chunk_tokens[:350]
            chunk_text = tokenizer.decode(
                chunk_tokens,
                skip_special_tokens=True
            ).strip()

        chunks.append(chunk_text)

        if end == len(tokens):
            break

        start = max(end - overlap_tokens, 0)

    return chunks


def split_by_article_boundaries(
    text,
    tokenizer,
    max_tokens=400,
    overlap_tokens=50
):

    text = clean_text(text)

    sections = BOUNDARY_RE.split(text)

    chunks = []

    for section in sections:

        section = section.strip()
        if not section:
            continue

        token_count = len(tokenizer.encode(section, add_special_tokens=False))

        if token_count <= max_tokens:
            chunks.append(section)

        else:
            chunks.extend(
                split_long_text_tokens(
                    section,
                    tokenizer=tokenizer,
                    max_tokens=max_tokens,
                    overlap_tokens=overlap_tokens
                )
            )

    return chunks

In [ ]:
def extract_metadata(section):

    metadata = {
        "type": None,
        "title": None,
        "chapter": None,
        "section": None,
        "article": None,
        "recital": None,
        "annex": None
    }

    m = TITLE_RE.search(section)
    if m:
        metadata["title"] = m.group(1)

    m = CHAPTER_RE.search(section)
    if m:
        metadata["chapter"] = m.group(1)

    m = SECTION_RE.search(section)
    if m:
        metadata["section"] = m.group(1)

    m = ARTICLE_RE.search(section)
    if m:
        metadata["type"] = "article"
        metadata["article"] = int(m.group(1))

    m = RECITAL_RE.search(section)
    if m:
        metadata["type"] = "recital"
        metadata["recital"] = int(m.group(1))

    m = ANNEX_RE.search(section)
    if m:
        metadata["type"] = "annex"
        metadata["annex"] = m.group(1)

    return metadata

In [ ]:
def extract_pdf_pages(pdf_path):

    doc = fitz.open(pdf_path)

    rows = []

    for i in range(len(doc)):

        page = doc.load_page(i)

        text = clean_text(
            page.get_text("text")
        )

        if text:

            rows.append(
                {
                    "doc_id": "eu_ai_act",
                    "page": i + 1,
                    "title": "EU AI Act",
                    "text": text
                }
            )

    return pd.DataFrame(rows)

In [ ]:
def build_article_chunks(
    pages_df,
    doc_id="eu_ai_act"
):

    full_text = ""
    page_offsets = []

    for _, row in pages_df.iterrows():

        page_offsets.append(
            (
                len(full_text),
                int(row["page"])
            )
        )

        full_text += row["text"] + "\n\n"

    def char_to_page(pos):

        page = page_offsets[0][1]

        for offset, pg in page_offsets:

            if offset <= pos:
                page = pg
            else:
                break

        return page

    sections = split_by_article_boundaries(
        full_text,
        tokenizer=tokenizer
    )

    chunks = []

    search_pos = 0

    current_title = None
    current_chapter = None
    current_section = None

    for i, section in enumerate(sections):

        idx = full_text.find(
            section[:80],
            search_pos
        )

        page = char_to_page(idx) if idx != -1 else 1

        search_pos = max(idx, 0)

        md = extract_metadata(section)

        if md["title"] is not None:
            current_title = md["title"]

        if md["chapter"] is not None:
            current_chapter = md["chapter"]

        if md["section"] is not None:
            current_section = md["section"]

        metadata = {
            "document": "EU AI Act",
            "page": page,
            "part": i,
            "type": md["type"],
            "title": current_title,
            "chapter": current_chapter,
            "section": current_section,
            "article": md["article"],
            "recital": md["recital"],
            "annex": md["annex"]
        }
        prefix = ""

        if current_title:
            prefix += f"Title {current_title}. "

        if current_chapter:
            prefix += f"Chapter {current_chapter}. "

        if md["article"]:
            prefix += f"Article {md['article']}. "

        enriched_text = prefix + section

        chunks.append(
            Chunk(
                chunk_id=f"{doc_id}_{i}",
                doc_id=doc_id,
                text=enriched_text,
                metadata=metadata
            )
        )

    return chunks

In [ ]:
pages_df = extract_pdf_pages(PDF_PATH)

chunks_article = build_article_chunks(
    pages_df
)

chunks = chunks_article

print(
    f"Article chunks: {len(chunks_article)}"
)

In [ ]:
def chunks_to_dataframe(chunk_list):

    rows = []

    for c in chunk_list:

        row = {
            "chunk_id": c.chunk_id,
            "doc_id": c.doc_id,
            "text": c.text
        }

        row.update(c.metadata)

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
chunks_article_df = chunks_to_dataframe(
    chunks_article
)

chunks_article_df.to_csv(
    os.path.join(
        PROCESSED_DIR,
        "chunks_article.csv"
    ),
    index=False
)

print(
    f"Saved {len(chunks_article_df)} chunks"
)

print("\n--- First 15 chunks ---")
display(chunks_article_df[
    [
        "chunk_id",
        "page",
        "chapter",
        "section",
        "article",
        "type"
    ]
].head(5))

print("\n--- Chunks with missing chapter/section metadata (first 5 examples) ---")
missing_metadata_chunks = chunks_article_df[
    chunks_article_df["chapter"].isnull() & chunks_article_df["section"].isnull()
].head(5)

if not missing_metadata_chunks.empty:
    for _, row in missing_metadata_chunks.iterrows():
        print(f"\nChunk ID: {row['chunk_id']}, Page: {row['page']}")
        print(f"Type: {row['type']}, Article: {row['article']}")
        print(f"Text (first 200 tok):\n{row['text'][:200]}...")
else:
    print("No chunks found with missing chapter/section metadata.")

### Embedding & Vector Stores

Two separate FAISS indexes and BM25 indexes are built — one per chunk set:

| Index | Chunk source | Used by |
|---|---|---|
| `faiss_index_fixed` + `bm25_fixed` | Fixed-size chunks | Baseline pipeline |
| `faiss_index_article` + `bm25_article` | Article-boundary chunks | Enhanced pipeline |

This ensures the comparison is fair: baseline uses its own index, enhanced uses its own. Both use `all-MiniLM-L6-v2` (384-dim, normalised) for embeddings.


In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

def bm25_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

def build_faiss_index(chunk_list):
    texts = [c.text for c in chunk_list]

    embeddings = embed_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    dim = embeddings.shape[1]

    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    # IMPORTANT: mapping FAISS index → chunk
    id_map = {i: chunk_list[i] for i in range(len(chunk_list))}

    return index, id_map, dim

def build_bm25_index(chunk_list):
    tokenized = [bm25_tokenize(c.text) for c in chunk_list]
    return BM25Okapi(tokenized)

def fixed_size_chunker(text, chunk_size=1800, chunk_overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += (chunk_size - chunk_overlap)
        if start >= len(text):
            break
    return chunks

def build_fixed_chunks(pages_df, doc_id="eu_ai_act"):
    chunks = []
    chunk_counter = 0
    for _, row in pages_df.iterrows():
        page_text = row["text"]
        page_num = row["page"]

        page_chunks = fixed_size_chunker(page_text)

        for i, text_chunk in enumerate(page_chunks):
            metadata = {
                "document": "EU AI Act",
                "page": page_num,
                "part": i,
                "type": "fixed_size",
                "title": None,
                "chapter": None,
                "section": None,
                "article": None,
                "recital": None,
                "annex": None
            }
            chunks.append(
                Chunk(
                    chunk_id=f"{doc_id}_fixed_{chunk_counter}",
                    doc_id=doc_id,
                    text=text_chunk,
                    metadata=metadata
                )
            )
            chunk_counter += 1
    return chunks


print("Building indexes for FIXED-SIZE chunks (baseline)...")

chunks_fixed = build_fixed_chunks(pages_df)
print(f"Fixed-size chunks: {len(chunks_fixed)}")

faiss_index_fixed, faiss_map_fixed, dim = build_faiss_index(chunks_fixed)
bm25_fixed = build_bm25_index(chunks_fixed)


print("\nBuilding indexes for ARTICLE-BOUNDARY chunks (enhanced)...")

faiss_index_article, faiss_map_article, _ = build_faiss_index(chunks_article)
bm25_article = build_bm25_index(chunks_article)


print(f"\nFixed-size FAISS: {faiss_index_fixed.ntotal}")
print(f"Article-boundary FAISS: {faiss_index_article.ntotal}")

In [ ]:
def rrf_fusion(dense_ids, sparse_ids, k=60):
    """
    Reciprocal Rank Fusion for hybrid retrieval
    """

    scores = {}

    # Dense results
    for rank, idx in enumerate(dense_ids):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    # Sparse results
    for rank, idx in enumerate(sparse_ids):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    # Sort by score
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [idx for idx, _ in ranked]

In [ ]:
def retrieve_hybrid(query, k_dense=10, k_sparse=10, use="article"):

    if use == "article":
        faiss_index = faiss_index_article
        faiss_map = faiss_map_article
        bm25 = bm25_article
        chunks = chunks_article
    else:
        faiss_index = faiss_index_fixed
        faiss_map = faiss_map_fixed
        bm25 = bm25_fixed
        chunks = chunks_fixed

    # --- Dense retrieval (FAISS) ---
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = faiss_index.search(q_emb, k_dense)
    dense_ids = indices[0].tolist()

    # --- Sparse retrieval (BM25) ---
    tokenized_query = bm25_tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)
    sparse_ids = np.argsort(bm25_scores)[::-1][:k_sparse].tolist()

    # --- Fusion ---
    fused_ids = rrf_fusion(dense_ids, sparse_ids)

    # --- Return chunks ---
    results = [chunks[i] for i in fused_ids[:max(k_dense, k_sparse)]]

    return results

In [ ]:
reranker = CrossEncoder(RERANK_MODEL_NAME, max_length=512)
def rerank(query, candidates, top_k=5):

    pairs = [(query, c["chunk"].text) for c in candidates]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        {
            "chunk": item[0]["chunk"],
            "score": float(item[1])
        }
        for item in ranked[:top_k]
    ]

In [ ]:
def retrieve_and_rerank(query, use="article", top_k=5):

    # Step 1: Hybrid retrieval (FAISS + BM25 + RRF)
    raw_candidates = retrieve_hybrid(query, use=use)

    # Transform raw_candidates into the format expected by the current rerank function
    candidates_for_rerank = [{"chunk": c, "score": 1.0} for c in raw_candidates]

    # Step 2: Cross-encoder reranking
    top_chunks = rerank(query, candidates_for_rerank, top_k=top_k)

    expanded = []

    seen_articles = set()

    for item in top_chunks:

        chunk = item['chunk']

        article = chunk.metadata["article"]

        if article is None:
            expanded.append(item)
            continue

        if article not in seen_articles:

            article_chunks = [

                c for c in chunks_article

                if c.metadata["article"] == article
            ]

            for c in article_chunks:

                expanded.append(
                    {
                        "chunk": c,
                        "score": item["score"]
                    }
                )

            seen_articles.add(article)

    return expanded

In [ ]:
def retrieve_enhanced(query, model_override=None):

    bm25_query = rule_based_rewrite(query)

    bm25_hits = bm25_retrieve(
        bm25_query,
        chunks_article,
        bm25_article,
        top_k=TOP_K_BM25
    )

    # Optimized: generate a single HyDE from the original query instead of subqueries.
    # Removed: queries = generate_subqueries(query)

    # Removed loop: for q in queries:
    hyde_q = hyde_rewrite(query) # Now only one HyDE call for the original query

    dense_hits = dense_retrieve(
        hyde_q,
        chunks_article,
        faiss_index_article,
        top_k=TOP_K_DENSE # Using TOP_K_DENSE instead of fixed 15
    )

    fused = reciprocal_rank_fusion(
        [bm25_hits, dense_hits]
    )

    reranked = rerank(
        query,
        fused[:TOP_K_FUSION],
        top_k=TOP_K_RERANK
    )

    return reranked, hyde_q

In [ ]:
query = "What are the prohibited AI practices?"

results = retrieve_and_rerank(query, use="article")

for r in results:
    print("\n---")
    print(r["chunk"].metadata)
    print(r["chunk"].text[:300])

## 4. Advanced RAG Pipeline

### Advanced Improvements Implemented

| Improvement | Description |
|---|---|
| **Article-boundary chunking** | Splits the document at Article / Recital / Annex headings so each chunk contains exactly one self-contained legal provision, eliminating mid-article breaks that cause incomplete retrieval |
| **Hybrid retrieval (BM25 + Dense)** | BM25 captures exact legal term matches; dense retrieval captures semantic similarity. Both are required for legal text |
| **Reciprocal Rank Fusion (RRF)** | Fuses the two ranked lists without requiring score normalisation |
| **Cross-encoder reranking** | `ms-marco-MiniLM-L-6-v2` jointly re-scores query–chunk pairs for higher precision than bi-encoder cosine similarity alone |
| **Rule-based query rewriting** | Normalises informal legal shorthand ("AI Act" → "Regulation (EU) 2024/1689"; "high risk" → "high-risk AI system") |
| **HyDE (Hypothetical Document Embeddings)** | Generates a hypothetical answer in the style of the Act and uses its embedding as the dense retrieval vector, bridging the vocabulary gap between user queries and legal text |

### Baseline vs. Enhanced Pipelines

| Stage | Baseline | Enhanced |
|---|---|---|
| Chunking | Fixed-size (1 800 chars) | Article-boundary |
| Index | `faiss_index_fixed` + `bm25_fixed` | `faiss_index_article` + `bm25_article` |
| Query | Raw user query | Rewritten + HyDE hypothetical |
| Retrieval | Dense (FAISS) only | BM25 + HyDE-Dense + RRF |
| Re-ranking | None | Cross-encoder (top-20 → top-5) |
| Generation | Gemini + context | Gemini + context |


In [ ]:
RULE_MAP = {
    "ai act": "AI Act Regulation (EU) 2024/1689",
    "eu ai act": "EU AI Act Regulation (EU) 2024/1689",
    "high risk": "high-risk AI system",
    "prohibited": "prohibited artificial intelligence practices",
    "ban": "prohibited artificial intelligence practices"
}

def rule_based_rewrite(query):

    q = query.lower()

    for k,v in RULE_MAP.items():

        if re.search(rf"\b{k}\b", q):

            q += " " + v

    return q

In [ ]:
hyde_cache = {}

def hyde_rewrite(query, model_override=None):

    if query in hyde_cache:
        return hyde_cache[query]

    prompt = """
You are an expert legal assistant.

Write a short factual passage that could appear in the EU AI Act and would answer:

{query}

Return only the passage.
""".format(query=query)

    model_to_use = model_override if model_override else GEMINI_MODEL
    model = genai.GenerativeModel(model_to_use)

    response = model.generate_content(prompt)
    time.sleep(15) # Add a delay after each API call

    hyde_text = response.text.strip()

    hyde_cache[query] = hyde_text

    return hyde_text

In [ ]:
def prepare_query(query, mode="rule"):
    """
    mode:
    - rule
    - hyde
    """

    if mode == "hyde":
        rewritten = hyde_rewrite(query)
    else:
        rewritten = rule_based_rewrite(query)

    return rewritten

In [ ]:
def generate_subqueries(query):

    prompt = f"""
Generate three alternative formulations of the following legal question.

Question:
{query}

Return only the three questions.
"""

    model = genai.GenerativeModel(GEMINI_MODEL)

    response = model.generate_content(prompt)
    time.sleep(15)

    lines = [
        x.strip("-123456789. ")
        for x in response.text.split("\n")
        if len(x.strip()) > 0
    ]

    return [query] + lines[:3]

In [ ]:
def bm25_retrieve(query, target_chunks, target_bm25_object, top_k=10):

    tokens = bm25_tokenize(query)

    scores = target_bm25_object.get_scores(tokens)

    idxs = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in idxs:
        results.append({
            "chunk": target_chunks[idx],
            "score": float(scores[idx]),
            "method": "bm25"
        })

    return results

In [ ]:
def dense_retrieve(query, target_chunks, target_faiss_index, top_k=10):

    q_emb = embed_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, idxs = target_faiss_index.search(q_emb, top_k)

    results = []

    for score, idx in zip(scores[0], idxs[0]):

        results.append({
            "chunk": target_chunks[idx],
            "score": float(score),
            "method": "dense"
        })

    return results

In [ ]:
def reciprocal_rank_fusion(
    result_lists,
    weights=[1.0, 2.0],
    k=60
):

    fused = {}
    chunk_map = {}

    # Ensure weights match the number of result_lists, or use a default if mismatch
    if len(weights) != len(result_lists):
        weights = [1.0] * len(result_lists)

    for list_idx, res_list in enumerate(result_lists):
        current_weight = weights[list_idx]

        for rank, item in enumerate(res_list, start=1):
            cid = item["chunk"].chunk_id
            fused[cid] = fused.get(cid, 0.0) + current_weight / (k + rank)
            chunk_map[cid] = item["chunk"]

    merged = [
        {
            "chunk": chunk_map[cid],
            "score": score
        }
        for cid, score in fused.items()
    ]

    merged.sort(key=lambda x: x["score"], reverse=True)

    return merged

In [ ]:
def rerank(query, candidates, top_k=5):

    pairs = [(query, c["chunk"].text) for c in candidates]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        {
            "chunk": item[0]["chunk"],
            "score": float(item[1])
        }
        for item in ranked[:top_k]
    ]

In [ ]:
def retrieve_baseline(query):

    return dense_retrieve(
        query,
        chunks_fixed,
        faiss_index_fixed,
        top_k=TOP_K_DENSE
    )

## 5. Generation with Retrieved Context

### Prompt Engineering

The prompt instructs the model to:
- Act as an expert EU legal assistant
- Answer **only** from the provided context (grounding constraint)
- Explicitly say "I do not have enough evidence" if the context is insufficient (reduces hallucination)
- Cite page numbers in the answer (verifiability)

This is a strict "closed-book" prompt pattern appropriate for compliance use cases where hallucinated legal claims are unacceptable.


In [ ]:
def build_context(hits):

    blocks = []

    for i, item in enumerate(hits, start=1):

        chunk = item["chunk"]

        page = chunk.metadata.get("page")

        article = chunk.metadata.get("article")

        location = f"page {page}"

        if article:
            location += f", Article {article}"

        blocks.append(
            f"[{i}] {location}\n{chunk.text}"
        )

    return "\n\n".join(blocks)

In [ ]:
def build_prompt(query, retrieved_chunks):

    context = build_context(
        [{"chunk": c} for c in retrieved_chunks]
    )

    prompt_template = """
You are an expert EU legal assistant for the EU AI Act.

Answer the user's question ONLY using the provided context. If the context does not contain enough information to answer the question, state "I do not have enough evidence from the provided context to answer this question." Do not make up any information.

Cite the page numbers of the context where you found the information. If a chunk has an Article number, also cite the Article.

Question: {query}

Context:
{context}

Answer:"""

    return prompt_template.format(query=query, context=context)

In [ ]:
def generate_with_gemini(prompt, model_override=None):

    model_to_use = model_override if model_override else GEMINI_MODEL
    model = genai.GenerativeModel(model_to_use)

    try:
        response = model.generate_content(prompt)
        time.sleep(15) # Add a delay after each API call
        return response.text.strip()
    except Exception as e:
        # Handle cases where the model might block content or fail to generate
        return f"Error generating content: {e}"

In [ ]:
def run_baseline(query, model_override=None):

    hits = retrieve_baseline(query)

    retrieved_chunks = [item["chunk"] for item in hits]

    prompt = build_prompt(query, retrieved_chunks)

    answer = generate_with_gemini(prompt, model_override=model_override)

    context_string = build_context(hits)

    return {
        "answer": answer,
        "hits": hits,
        "retrieved": retrieved_chunks,
        "context": context_string
    }

In [ ]:
def run_enhanced(query, model_override=None):

    bm25_query = rule_based_rewrite(query)

    bm25_hits = bm25_retrieve(
        bm25_query,
        chunks_article,
        bm25_article,
        top_k=TOP_K_BM25
    )

    hyde_q = hyde_rewrite(query)

    dense_hits = dense_retrieve(
        hyde_q,
        chunks_article,
        faiss_index_article,
        top_k=TOP_K_DENSE
    )

    fused = reciprocal_rank_fusion(
        [bm25_hits, dense_hits]
    )

    reranked = rerank(
        query,
        fused[:TOP_K_FUSION],
        top_k=TOP_K_RERANK
    )

    retrieved_chunks = [item["chunk"] for item in reranked]

    prompt = build_prompt(query, retrieved_chunks)

    answer = generate_with_gemini(prompt, model_override=model_override)

    context_string = build_context(reranked)

    return {
        "answer": answer,
        "hits": reranked,
        "retrieved": retrieved_chunks,
        "context": context_string,
        "rewritten": hyde_q # Store HyDE query for tracing if needed
    }

In [ ]:
eval_data = [
    {
        "query": "What is the purpose of the EU AI Act?",
        "gold_answer": """
The purpose of the EU AI Act is to improve the functioning of the internal market
and promote the uptake of human-centric and trustworthy AI while ensuring a high
level of protection of health, safety and fundamental rights.
""",
        "gold_pages": [44, 45],
        "category": "General Information"
    },

    {
        "query": "Which AI practices are prohibited under the EU AI Act?",
        "gold_answer": """
The EU AI Act prohibits certain AI practices that present unacceptable risks,
including manipulative practices, exploitation of vulnerabilities, social scoring,
and certain forms of biometric identification.
""",
        "gold_pages": [51, 52, 53],
        "category": "Prohibitions"
    },

    {
        "query": "A researcher develops an AI model exclusively for scientific research and does not place it on the market. Is the model covered by the AI Act?",
        "gold_answer": """
AI systems developed and used solely for scientific research and development are
excluded from the scope of the EU AI Act if they are not placed on the market or
put into service.
""",
        "gold_pages": [46],
        "category": "Edge Case"
    }
]

In [ ]:
import numpy as np

def mean_reciprocal_rank(retrieved_pages, gold_pages):

    gold_set = set(gold_pages)

    for i, p in enumerate(retrieved_pages, start=1):

        if p in gold_set:
            return 1.0 / i

    return 0.0


def ndcg_at_k(retrieved_pages, gold_pages, k=5):

    gold_set = set(gold_pages)

    dcg = 0

    for i, p in enumerate(retrieved_pages[:k], start=1):

        rel = 1 if p in gold_set else 0

        dcg += rel / np.log2(i + 1)

    ideal_hits = min(len(gold_set), k)

    idcg = sum(
        1 / np.log2(i + 1)
        for i in range(1, ideal_hits + 1)
    )

    return dcg / idcg if idcg > 0 else 0


def retrieval_metrics(retrieved_pages, gold_pages):

    gold_set = set(gold_pages)
    retrieved_set = set(retrieved_pages)

    hits = retrieved_set & gold_set

    hit_at_5 = int(len(hits) > 0)

    precision = len(hits) / len(retrieved_pages) if retrieved_pages else 0

    recall = len(hits) / len(gold_set) if gold_set else 0

    mrr = mean_reciprocal_rank(
        retrieved_pages,
        gold_pages
    )

    ndcg = ndcg_at_k(
        retrieved_pages,
        gold_pages,
        k=5
    )

    return {
        "hit@5": hit_at_5,
        "precision@5": precision,
        "recall@5": recall,
        "mrr": mrr,
        "ndcg@5": ndcg
    }

In [ ]:
import json
import time

def llm_judge_score(
        query,
        gold_answer,
        generated_answer,
        page_citations,
        model_override=None
):

    judge_prompt = f"""
You are a strict evaluator for answers about the EU AI Act.

Score the following dimensions:

Correctness
1 = incorrect
2 = partially correct
3 = fully correct

Grounding
1 = unsupported by retrieved context
2 = partially supported
3 = fully supported

Completeness
1 = major omissions
2 = minor omissions
3 = complete

Citation Quality
1 = wrong citations
2 = partially correct citations
3 = correct citations

Question:
{query}

Gold Answer:
{gold_answer}

Generated Answer:
{generated_answer}

Retrieved Pages:
{page_citations}

Return ONLY:

{{
"correctness":1,
"grounding":1,
"completeness":1,
"citation_quality":1
}}
"""

    model_to_use = (
        model_override
        if model_override
        else GEMINI_EVAL_MODEL
    )

    model = genai.GenerativeModel(model_to_use)

    try:

        response = model.generate_content(judge_prompt)

        time.sleep(15)

        return json.loads(response.text)

    except Exception as e:

        print(e)

        return {
            "correctness": None,
            "grounding": None,
            "completeness": None,
            "citation_quality": None
        }

In [ ]:
from sentence_transformers import SentenceTransformer

eval_embedder = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


def context_precision(
        retrieved_chunks,
        gold_answer
):

    context = " ".join(
        c.text.lower()
        for c in retrieved_chunks
    )

    gold_words = gold_answer.lower().split()

    hits = sum(
        1
        for w in gold_words
        if w in context
    )

    return hits / len(gold_words)


def context_recall(
        retrieved_chunks,
        gold_answer
):

    context = " ".join(
        c.text.lower()
        for c in retrieved_chunks
    )

    gold_words = set(
        gold_answer.lower().split()
    )

    hits = sum(
        1
        for w in gold_words
        if w in context
    )

    return hits / len(gold_words)


def answer_relevancy(
        query,
        answer
):

    q_emb = eval_embedder.encode(
        query,
        normalize_embeddings=True
    )

    a_emb = eval_embedder.encode(
        answer,
        normalize_embeddings=True
    )

    return float(np.dot(q_emb, a_emb))


def faithfulness(
        answer,
        retrieved_chunks
):

    context = " ".join(
        c.text.lower()
        for c in retrieved_chunks
    )

    answer_tokens = answer.lower().split()

    overlap = sum(
        1
        for token in answer_tokens
        if token in context
    )

    return overlap / len(answer_tokens)


def evaluate_ragas_light(
        query,
        answer,
        retrieved_chunks,
        gold_answer
):

    return {

        "context_precision":
            context_precision(
                retrieved_chunks,
                gold_answer
            ),

        "context_recall":
            context_recall(
                retrieved_chunks,
                gold_answer
            ),

        "answer_relevancy":
            answer_relevancy(
                query,
                answer
            ),

        "faithfulness":
            faithfulness(
                answer,
                retrieved_chunks
            )
    }

In [ ]:
from tqdm.auto import tqdm

rows = []

for item in tqdm(eval_data):

    query = item["query"]

    gold_answer = item["gold_answer"]

    gold_pages = item["gold_pages"]

    category = item["category"]

    base_res = run_baseline(
        query,
        model_override=GEMINI_EVAL_MODEL
    )

    enh_res = run_enhanced(
        query,
        model_override=GEMINI_EVAL_MODEL
    )

    base_pages = [
        c.metadata["page"]
        for c in base_res["retrieved"]
    ]

    enh_pages = [
        c.metadata["page"]
        for c in enh_res["retrieved"]
    ]

    base_ret = retrieval_metrics(
        base_pages,
        gold_pages
    )

    enh_ret = retrieval_metrics(
        enh_pages,
        gold_pages
    )

    base_judge = llm_judge_score(
        query,
        gold_answer,
        base_res["answer"],
        base_pages
    )

    enh_judge = llm_judge_score(
        query,
        gold_answer,
        enh_res["answer"],
        enh_pages
    )

    base_ragas = evaluate_ragas_light(
        query,
        base_res["answer"],
        base_res["retrieved"],
        gold_answer
    )

    enh_ragas = evaluate_ragas_light(
        query,
        enh_res["answer"],
        enh_res["retrieved"],
        gold_answer
    )

    rows.append({

        "query": query,
        "category": category,

        **{
            f"baseline_{k}": v
            for k, v in base_ret.items()
        },

        **{
            f"enhanced_{k}": v
            for k, v in enh_ret.items()
        },

        **{
            f"baseline_{k}": v
            for k, v in base_judge.items()
        },

        **{
            f"enhanced_{k}": v
            for k, v in enh_judge.items()
        },

        **{
            f"baseline_{k}": v
            for k, v in base_ragas.items()
        },

        **{
            f"enhanced_{k}": v
            for k, v in enh_ragas.items()
        }
    })

evaluation_df = pd.DataFrame(rows)

evaluation_df

In [ ]:
metrics = [

    "hit@5",
    "precision@5",
    "recall@5",
    "mrr",
    "ndcg@5",

    "correctness",
    "grounding",
    "completeness",
    "citation_quality",

    "context_precision",
    "context_recall",
    "answer_relevancy",
    "faithfulness"
]

print("=" * 80)
print("FINAL EVALUATION SUMMARY")
print("=" * 80)

for metric in metrics:

    baseline_score = (
        evaluation_df[
            f"baseline_{metric}"
        ]
        .mean()
    )

    enhanced_score = (
        evaluation_df[
            f"enhanced_{metric}"
        ]
        .mean()
    )

    print(
        f"{metric:<20}"
        f"| baseline: {baseline_score:.3f} "
        f"| enhanced: {enhanced_score:.3f}"
    )

### Description of Results

The evaluation summary presents a mixed performance between the 'baseline' and 'enhanced' RAG pipelines across various metrics.


The baseline pipeline generally outperforms the enhanced pipeline in these direct retrieval metrics. For instance, `hit@5` (1.000 vs 0.667), `recall@5` (0.889 vs 0.500), `mrr` (0.694 vs 0.444), and `ndcg@5` (0.538 vs 0.436) are all higher for the baseline.

Meanwhile, in the LLM Judge Metrics (correctness, grounding, completeness, citation_quality)enhanced pipeline significantly outperforms the baseline in `correctness` (3.000 vs 2.667) and `completeness` (3.000 vs 2.333). `Grounding` is equal for both. However, `citation_quality` is better for the baseline (2.333 vs 2.000).

RAGAS-Light Metrics (context_precision, context_recall, answer_relevancy, faithfulness) shows that the enhanced pipeline shows better `context_precision` (0.927 vs 0.903), `context_recall` (0.915 vs 0.889), and `faithfulness` (0.831 vs 0.784). `Answer_relevancy` is slightly better for the baseline.


### Explanation of Results:

The contrasting results suggest that while the enhanced pipeline's sophisticated retrieval mechanisms (HyDE, RRF, Reranking, Article-boundary chunking) are effective at providing a richer and more relevant context to the LLM for generation, they don't always translate into superior raw 'top-k page retrieval' metrics on this small evaluation set.
The unexpected lower performance of the enhanced pipeline in `hit@5`, `precision@5`, `recall@5`, `mrr`, and `ndcg@5` is likely due to the small evaluation dataset. A single query where the enhanced system fails to place a gold page within the top 5 retrieved items would significantly pull down these averages. It could also be influenced by Article Expansion Logic. The `retrieve_and_rerank` function expands the context to include all chunks of an article if any part of it is retrieved. While good for completeness, if the 'gold pages' for evaluation are very specific and not all parts of the expanded article are considered 'gold', this could dilute precision-focused metrics.

The `top_k` values in various retrieval stages might not be optimally tuned for this specific dataset and gold page definition, leading to some gold pages being ranked just outside the strict top-k cutoffs used for these metrics.

The enhanced pipeline's strength lies in its ability to empower the LLM. The higher `correctness`, `completeness`, and `faithfulness` scores indicate that the context ultimately provided to the LLM (which includes the benefits of article-boundary chunking, hybrid retrieval, and reranking) is of superior quality, leading to more accurate, comprehensive, and well-grounded answers.

The improved `context_precision` and `context_recall` suggests that the enhanced pipeline is better at retrieving *all* the relevant information needed for the answer and that a higher proportion of what it retrieves is actually relevant.

The slightly lower `citation_quality` for the enhanced pipeline might be related to the article expansion. If the LLM is citing pages from the expanded context that are accurate but not strictly part of the narrowly defined 'gold pages' for a query, it could penalize this metric.


### How to Enhance It Further:

There are few things that can be added/performed to improve the performance of this RAG system, such are:
1. Expanding the evaluation dataset to include more diverse queries for stronger and more reliable performance measurement;
2. Refining gold standard labels (gold_pages), to ensure accurate coverage of relevant content and better handling of partial relevance in expanded articles.
3. Tuning retrieval hyperparameters (TOP_K_BM25, TOP_K_DENSE, TOP_K_FUSION, TOP_K_RERANK) using systematic methods such as grid or Bayesian search.
4. Reassessing article expansion logic to reduce noise, potentially using confidence-based or selective expansion strategies.
5. Improving LLM prompt design to enforce stricter, more precise citation grounding and reduce over-reliance on broad context.
6. Conducting targeted error analysis on low-performing queries to identify retrieval gaps and ranking weaknesses.

## 6 Demo Log: Sample Query

Here we examine how the baseline RAG system responds to a few sample queries. We first print the query itself. Then, call run_baseline with the demo_query to execute the entire baseline RAG pipeline. Finally, we print the answer generated by the baseline RAG system and the context that was retrieved to formulate that answer.

**Demo Query 1** : What is the purpose of the EU AI Act?

In [ ]:
demo_query = "What is the purpose of the EU AI Act?"

print(f"Query: {demo_query}\n")

baseline_output = run_baseline(demo_query)

print("--- Baseline RAG Answer ---")
print(baseline_output["answer"])

print("\n--- Retrieved Context (Baseline) ---")
print(baseline_output["context"])

The baseline system provides a concise answer, citing page numbers based on its fixed-size chunk retrieval. The answer is generally correct but might lack the depth or specific article references found in more granular chunks.

In [ ]:
enhanced_output = run_enhanced(demo_query)

print(f"Query: {demo_query}\n")

print("--- Enhanced RAG Answer ---")
print(enhanced_output["answer"])

print("\n--- Retrieved Context (Enhanced) ---")
print(enhanced_output["context"])

print(f"\n--- Rewritten Query (HyDE) ---")
print(enhanced_output["rewritten"])

The enhanced system's answer seems to be more precise and provides more specific citations, thanks to the article-boundary chunking, hybrid retrieval, and re-ranking mechanisms. The rewritten query (HyDE) also demonstrates how the system attempts to bridge the lexical gap between the user's input and the dense legal text.

Both options give correct answers, however the enhanced answer is more specific, targeted and provided better sourcing.

**Demo query 2** "Which AI practices are prohibited under the EU AI Act?"

In [ ]:
demo_query_2 = "Which AI practices are prohibited under the EU AI Act?"

print(f"Query: {demo_query_2}\n")

baseline_output_2 = run_baseline(demo_query_2)

print("--- Baseline RAG Answer ---")
print(baseline_output_2["answer"])

print("\n--- Retrieved Context (Baseline) ---")
print(baseline_output_2["context"])

The baseline system correctly identified and listed several prohibited practices, such as subliminal techniques, exploitation of vulnerabilities, and evaluation/classification of natural persons, citing Article 5 on page 51.

In [ ]:
enhanced_output_2 = run_enhanced(demo_query_2)

print(f"Query: {demo_query_2}\n")

print("--- Enhanced RAG Answer ---")
print(enhanced_output_2["answer"])

print("\n--- Retrieved Context (Enhanced) ---")
print(enhanced_output_2["context"])

print(f"\n--- Rewritten Query (HyDE) ---")
print(enhanced_output_2["rewritten"])

The enhanced system also identified and listed prohibited practices, explicitly citing Article 5 on pages 1 and 6. The answer from the enhanced system was generally more comprehensive, pulling details from different sections of the document, as expected due to its more sophisticated retrieval.
The HyDE query provided a very detailed, hypothetical answer describing various prohibited practices, which likely helped the dense retriever accurately identify relevant sections.

Both systems performed well in identifying prohibited practices. The enhanced system, with its article-boundary chunking and hybrid retrieval, provided a slightly more detailed answer and better context, demonstrating its ability to consolidate information across relevant articles.

**Demo query 3**: "A researcher develops an AI model exclusively for scientific research and does not place it on the market. Is the model covered by the AI Act?"

In [ ]:
demo_query_3 = "A researcher develops an AI model exclusively for scientific research and does not place it on the market. Is the model covered by the AI Act?"

print(f"Query: {demo_query_3}\n")

baseline_output_3 = run_baseline(demo_query_3)

print("--- Baseline RAG Answer ---")
print(baseline_output_3["answer"])

print("\n--- Retrieved Context (Baseline) ---")
print(baseline_output_3["context"])

The baseline system correctly answered that the model is not covered, citing page 46 (Article 2(6)) and page 50 (Recital 63), which refer to exemptions for scientific research and development, and also models not yet placed on the market.

In [ ]:
enhanced_output_3 = run_enhanced(demo_query_3)

print(f"Query: {demo_query_3}\n")

print("--- Enhanced RAG Answer ---")
print(enhanced_output_3["answer"])

print("\n--- Retrieved Context (Enhanced) ---")
print(enhanced_output_3["context"])

print(f"\n--- Rewritten Query (HyDE) ---")
print(enhanced_output_3["rewritten"])

The enhanced system also correctly stated that the model is not covered(although, grammatically, gives a wierd answer, by starting with "Yes" and then stating the model is not cover by the EU AI act), citing Article 59 on page 46, and also provided additional context from page 1 and page 4 about AI models for research, development, and prototyping activities.
The HyDE query accurately constructed a hypothetical regulation text about AI systems developed for scientific research not being covered by the Act under certain conditions.

Both systems accurately identified the exemption. However, the enhanced system provided a richer context with multiple citations that reinforce the exemption, which is very important for nuanced legal queries. This shows the value of its chunking strategy that keeps related legal provisions together.

**Demo query 4**: "What are the requirements for high-risk AI systems?"

In [ ]:
demo_query_4 = "What are the requirements for high-risk AI systems?"

print(f"Query: {demo_query_4}\n")

baseline_output_4 = run_baseline(demo_query_4)

print("--- Baseline RAG Answer ---")
print(baseline_output_4["answer"])

print("\n--- Retrieved Context (Baseline) ---")
print(baseline_output_4["context"])

The baseline system gave a comprehensive answer detailing various requirements, including risk management, data governance, quality management, technical documentation, transparency, human oversight, accuracy, robustness, cybersecurity, accessibility, and conformity assessment, with citations to multiple pages and articles.

In [ ]:
enhanced_output_4 = run_enhanced(demo_query_4)

print(f"Query: {demo_query_4}\n")

print("--- Enhanced RAG Answer ---")
print(enhanced_output_4["answer"])

print("\n--- Retrieved Context (Enhanced) ---")
print(enhanced_output_4["context"])

print(f"\n--- Rewritten Query (HyDE) ---")
print(enhanced_output_4["rewritten"])

The enhanced system also provided a detailed list of requirements for high-risk AI systems, including risk management, data and data governance, technical documentation, transparency, human oversight, accuracy, robustness, and cybersecurity. It cited pages 1, 55, 59, and 62, and specific articles like Article 10, Article 13, Article 8, and Article 16.
The HyDE query created a very detailed and structured hypothetical answer outlining the key requirements for high-risk AI systems.

For this complex query, both systems provided extensive and accurate information. The enhanced system's ability to fuse and re-rank results from different retrieval methods likely contributed to its detailed and well-cited answer, ensuring all critical aspects of high-risk AI system requirements are covered.

### 7 Qualitative Error Analysis and Retrieval Trace Inspection

This section was designed to provide insight into how the enhanced RAG system processes a query and retrieves relevant information.

In [ ]:
def show_low_performing_queries(threshold=6):

    low_df = enhanced_results_df[
        enhanced_results_df["total_score"] < threshold
    ]

    if low_df.empty:
        print("No low-performing queries found.")
        return

    print(f"\nFound {len(low_df)} low-performing queries:\n")

    for _, row in low_df.iterrows():

        print("=" * 100)
        print(f"QUERY: {row['query']}")
        print(f"CATEGORY: {row['category']}")
        print(f"TOTAL SCORE: {row['total_score']}")

        print("\nGOLD ANSWER:")
        print(row["gold_answer"])

        print("\nENHANCED ANSWER:")
        print(row["answer"])

        print("\nREWRITTEN QUERY:")
        print(row["rewritten_query"])

        print("\nRETRIEVED PAGES:")
        print(row["retrieved_pages"])

        print("=" * 100)

In [ ]:
def trace_retrieval(query, gold_pages=None):

    print("\n" + "=" * 120)
    print(f"QUERY: {query}")
    print("=" * 120)

    rewritten = rule_based_rewrite(query)

    print(f"\nRewritten Query:\n{rewritten}")

    # BM25
    bm25_hits = bm25_retrieve(
        rewritten,
        chunks_article,
        bm25_article,
        top_k=TOP_K_BM25
    )

    print("\n--- BM25 RESULTS ---")
    for i, h in enumerate(bm25_hits):
        page = h["chunk"].metadata.get("page")
        mark = " (GOLD)" if gold_pages and page in gold_pages else ""
        print(f"{i+1}. Page {page} | Score {h['score']:.2f}{mark}")
        print(h["chunk"].text[:120], "\n")

    # Dense (HyDE optional)
    hyde_query = prepare_query(query, mode="hyde")

    dense_hits = dense_retrieve(
        hyde_query,
        chunks_article,
        faiss_index_article,
        top_k=TOP_K_DENSE
    )

    print("\n--- DENSE (HyDE) RESULTS ---")
    for i, h in enumerate(dense_hits):
        page = h["chunk"].metadata.get("page")
        mark = " (GOLD)" if gold_pages and page in gold_pages else ""
        print(f"{i+1}. Page {page} | Score {h['score']:.3f}{mark}")
        print(h["chunk"].text[:120], "\n")

    # Fusion
    fused = reciprocal_rank_fusion([bm25_hits, dense_hits])

    print("\n--- RRF FUSION RESULTS ---")
    for i, h in enumerate(fused[:10]):
        page = h["chunk"].metadata.get("page")
        mark = " (GOLD)" if gold_pages and page in gold_pages else ""
        print(f"{i+1}. Page {page} | Score {h['score']:.4f}{mark}")
        print(h["chunk"].text[:120], "\n")

    # Rerank
    reranked = rerank(rewritten, fused[:20], top_k=TOP_K_RERANK)

    print("\n--- CROSS-ENCODER RERANKED ---")
    for i, h in enumerate(reranked):
        page = h["chunk"].metadata.get("page")
        mark = " (GOLD)" if gold_pages and page in gold_pages else ""
        print(f"{i+1}. Page {page} | Score {h['score']:.4f}{mark}")
        print(h["chunk"].text[:120], "\n")

    return reranked

In [ ]:
query = eval_df.loc[0, "query"]
gold_pages = eval_df.loc[0, "gold_pages"]

trace_retrieval(query, gold_pages)

The `trace_retrieval()` function was used to examine each stage of the retrieval pipeline, including query rewriting, BM25 retrieval, dense retrieval with HyDE, Reciprocal Rank Fusion (RRF), and cross-encoder re-ranking. This enabled a detailed analysis of how documents were retrieved and ranked throughout the process.

For the query *“What is the purpose of the EU AI Act?”*, BM25 retrieved several relevant chunks, although the target pages were not consistently ranked at the top. In contrast, dense retrieval with HyDE placed the gold pages (44 and 45) in the highest positions, demonstrating strong semantic understanding. The subsequent RRF and cross-encoder stages preserved and further refined these results, ultimately ranking the most relevant chunk first and highlighting the effectiveness of the multi-stage retrieval approach.
